# Entraînement Sécurisé MobileNetV2 - Google Colab

Ce notebook permet d'entraîner le modèle **sécurisé avec Adversarial Training** sur Google Colab.

## Différences avec le Baseline
- ✨ **Adversarial Training**: Génération de FGSM + PGD pendant l'entraînement
- 🛡️ **Robustesse**: Protection contre les attaques adversariales
- 📊 **Trade-off**: Accuracy standard vs robustesse adversariale

## Prérequis
1. Dataset préparé (train/val/test splits)
2. Runtime Colab avec GPU activé (Runtime > Change runtime type > GPU)

## Workflow
1. Installation des dépendances
2. Upload du dataset
3. Vérification de la structure
4. **Entraînement sécurisé avec adversarial training**
5. Téléchargement des résultats

## 1. Vérification du GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: GPU not available. Training will be VERY slow!")
    print("Enable GPU: Runtime > Change runtime type > GPU")

## 2. Installation des dépendances

Installation des packages nécessaires pour l'entraînement sécurisé.

In [ ]:
# Installation des dépendances principales
!pip install -q torch torchvision tqdm matplotlib pillow

In [ ]:
# Installation des dépendances de sécurité (Zone 1 & Zone 2)
!pip install -q scipy scikit-learn cryptography

### Upload du module de sécurité

**IMPORTANT**: Uploader le fichier `security_modules_colab.py` qui contient toutes les classes de sécurité Zone 1 & Zone 2.

Ce fichier se trouve dans votre projet local: `notebooks/security_modules_colab.py`

In [ ]:
# Upload du module de sécurité
from google.colab import files
print("📤 Veuillez uploader le fichier security_modules_colab.py...")
uploaded = files.upload()

# Vérifier que le fichier a été uploadé
if 'security_modules_colab.py' in uploaded:
    print("✅ Module de sécurité uploadé avec succès!")
    # Importer le module
    from security_modules_colab import (
        DataVerifier, PoisoningDetector, EncryptedStorage,
        DifferentialPrivacy, ModelSigner, EarlyStopping,
        apply_zone1_security, apply_zone2_post_training
    )
    print("✅ Toutes les classes de sécurité importées!")
else:
    print("❌ Fichier security_modules_colab.py non trouvé!")
    print("   Veuillez uploader le fichier depuis notebooks/security_modules_colab.py")

## 3. Upload du Dataset

### Option A: Upload manuel (recommandé pour dataset < 500 MB)

**Structure requise pour le ZIP:**
```
prepared.zip
├── train/
│   ├── safe/       (699 images .jpg)
│   └── dangerous/  (699 images .jpg)
├── val/
│   ├── safe/       (147 images .jpg)
│   └── dangerous/  (147 images .jpg)
└── test/
    ├── safe/       (102 images .jpg)
    └── dangerous/  (102 images .jpg)
```

**Instructions:**
1. Depuis votre projet local, compresser le dossier `data/prepared/` en ZIP
2. Nommer le fichier `prepared.zip`
3. Exécuter la cellule ci-dessous et sélectionner le fichier

In [ ]:
from google.colab import files
import zipfile
import os

print("📤 Upload du fichier prepared.zip...")
uploaded = files.upload()

# Extraire le ZIP
print("\n📦 Extraction du dataset...")
with zipfile.ZipFile('prepared.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

print("\n✅ Dataset extrait dans /content/data/prepared/")

### Option B: Upload depuis Google Drive (recommandé pour dataset > 500 MB)

**Instructions:**
1. Uploader `prepared.zip` dans votre Google Drive
2. Exécuter la cellule ci-dessous
3. Autoriser l'accès à Google Drive
4. Modifier le chemin si nécessaire

In [ ]:
from google.colab import drive
import zipfile
import shutil

# Monter Google Drive
drive.mount('/content/drive')

# Copier le ZIP depuis Drive (modifier le chemin si nécessaire)
drive_zip_path = '/content/drive/MyDrive/prepared.zip'  # Modifier ce chemin

if os.path.exists(drive_zip_path):
    print(f"📋 Copie depuis Drive: {drive_zip_path}")
    shutil.copy(drive_zip_path, '/content/prepared.zip')
    
    # Extraire
    print("📦 Extraction du dataset...")
    with zipfile.ZipFile('/content/prepared.zip', 'r') as zip_ref:
        zip_ref.extractall('data')
    
    print("✅ Dataset extrait!")
else:
    print(f"❌ Fichier non trouvé: {drive_zip_path}")
    print("Vérifiez le chemin dans Google Drive")

## 4. Vérification de la structure du dataset

In [ ]:
import os
from pathlib import Path

def verify_dataset_structure(base_path='data/prepared'):
    """Vérifie que la structure du dataset est correcte"""
    base = Path(base_path)
    
    if not base.exists():
        print(f"❌ Dossier {base_path} non trouvé!")
        return False
    
    print("📂 Structure du dataset:\n")
    
    total_images = 0
    for split in ['train', 'val', 'test']:
        split_path = base / split
        if not split_path.exists():
            print(f"❌ {split}/ manquant!")
            continue
            
        safe_count = len(list((split_path / 'safe').glob('*.jpg'))) if (split_path / 'safe').exists() else 0
        dangerous_count = len(list((split_path / 'dangerous').glob('*.jpg'))) if (split_path / 'dangerous').exists() else 0
        split_total = safe_count + dangerous_count
        total_images += split_total
        
        print(f"  {split}/")
        print(f"    safe: {safe_count} images")
        print(f"    dangerous: {dangerous_count} images")
        print(f"    Total: {split_total} images\n")
    
    print(f"\n✅ Total: {total_images} images")
    
    if total_images == 0:
        print("\n⚠️ ATTENTION: Aucune image trouvée!")
        return False
    
    return True

# Vérifier la structure
if verify_dataset_structure():
    print("\n🎉 Dataset correctement structuré!")
else:
    print("\n❌ Problème avec la structure du dataset.")
    print("Vérifiez que le ZIP contient bien train/val/test avec safe/ et dangerous/")

## 5. Code d'entraînement sécurisé

Définition des classes et fonctions pour l'entraînement avec **Adversarial Training**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List
import numpy as np
from pathlib import Path
from datetime import datetime


class DangerousObjectDataset(Dataset):
    """Dataset pour les images safe/dangerous avec support de quarantaine"""

    def __init__(self, data_dir: str, split: str, transform=None, excluded_files=None):
        """
        Args:
            data_dir: Chemin vers data/prepared/
            split: 'train', 'val' ou 'test'
            transform: Transformations à appliquer
            excluded_files: Liste de fichiers à exclure (quarantaine Zone 1)
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform

        self.images = []
        self.labels = []
        self.excluded_count = 0

        # Normaliser les fichiers exclus
        excluded_set = set()
        if excluded_files:
            excluded_set = {str(Path(f).resolve()) for f in excluded_files}

        # Charger les images depuis safe/ et dangerous/
        split_dir = self.data_dir / split

        # Classe safe (label 0)
        safe_dir = split_dir / 'safe'
        if safe_dir.exists():
            for img_path in safe_dir.glob('*.jpg'):
                if str(img_path.resolve()) not in excluded_set:
                    self.images.append(str(img_path))
                    self.labels.append(0)
                else:
                    self.excluded_count += 1

        # Classe dangerous (label 1)
        dangerous_dir = split_dir / 'dangerous'
        if dangerous_dir.exists():
            for img_path in dangerous_dir.glob('*.jpg'):
                if str(img_path.resolve()) not in excluded_set:
                    self.images.append(str(img_path))
                    self.labels.append(1)
                else:
                    self.excluded_count += 1

        print(f"  {split}: {len(self.images)} images "
              f"(safe: {self.labels.count(0)}, dangerous: {self.labels.count(1)})")
        if self.excluded_count > 0:
            print(f"    ⚠️  Exclus (quarantaine): {self.excluded_count}")

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path = self.images[idx]
        label = self.labels[idx]

        # Charger l'image
        image = Image.open(img_path).convert('RGB')

        # Appliquer les transformations
        if self.transform:
            image = self.transform(image)

        return image, label

### Zone 1: Modules de Sécurité des Données

Avant l'entraînement, nous intégrons les mesures de sécurité de Zone 1:
- **DataVerifier**: Tests statistiques sur la qualité des données
- **PoisoningDetector**: Détection d'empoisonnement par clustering DBSCAN
- **Quarantine automatique**: Isolation des échantillons suspects

In [ ]:
class MobileNetV2Classifier(nn.Module):
    """Classificateur basé sur MobileNetV2 - IDENTIQUE AU BASELINE"""

    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()

        # Charger MobileNetV2 pré-entraîné sur ImageNet
        self.mobilenet = models.mobilenet_v2(pretrained=pretrained)

        # Freeze early layers (transfer learning)
        for param in self.mobilenet.features[:-3].parameters():
            param.requires_grad = False

        # Remplacer le classifier
        in_features = self.mobilenet.classifier[1].in_features

        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.45),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.45),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mobilenet(x)




### Générateur d'Exemples Adversariaux

Cette classe génère des exemples adversariaux pendant l'entraînement:
- **FGSM** (Fast Gradient Sign Method): Attaque rapide
- **PGD** (Projected Gradient Descent): Attaque itérative plus forte

In [ ]:
class AdversarialExampleGenerator:
    """Générateur d'exemples adversariaux pour Adversarial Training"""

    def __init__(self, model, criterion, epsilon=0.1):
        self.model = model
        self.criterion = criterion
        self.epsilon = epsilon

    def generate_fgsm(self, data, labels):
        """Génère des exemples adversariaux FGSM"""
        data.requires_grad = True
        output = self.model(data)
        loss = self.criterion(output, labels)

        self.model.zero_grad()
        loss.backward()

        # FGSM perturbation
        data_grad = data.grad.data
        sign_data_grad = data_grad.sign()
        fgsm_data = data + self.epsilon * sign_data_grad
        fgsm_data = torch.clamp(fgsm_data, 0, 1)

        return fgsm_data.detach()

    def generate_pgd(self, data, labels, num_steps=2, step_size=0.01):
        """Génère des exemples adversariaux PGD (itératif)"""
        # Initialisation aléatoire
        pgd_data = data.clone().detach()
        pgd_data = pgd_data + torch.empty_like(pgd_data).uniform_(-self.epsilon, self.epsilon)
        pgd_data = torch.clamp(pgd_data, 0, 1).detach()

        # Itérations PGD
        for _ in range(num_steps):
            pgd_data.requires_grad = True
            output = self.model(pgd_data)
            loss = self.criterion(output, labels)

            loss.backward()
            pgd_data = pgd_data + step_size * pgd_data.grad.sign()
            pgd_data = torch.clamp(pgd_data, data - self.epsilon, data + self.epsilon)
            pgd_data = torch.clamp(pgd_data, 0, 1).detach()

        return pgd_data


## 6. Configuration et lancement de l'entraînement sécurisé

In [ ]:
# Configuration FGSM OPTIMIZED - Pour maximiser robustesse FGSM
DATA_DIR = 'data/prepared'
OUTPUT_DIR = 'models/secured'
BATCH_SIZE = 32
NUM_EPOCHS = 30                        # Réduit à 30 epochs
LEARNING_RATE = 0.0001                 # Maintenu faible pour stabilité
WEIGHT_DECAY = 1e-4

# Configuration Adversarial Training - FGSM OPTIMIZED
ADVERSARIAL_TRAINING = True
EPSILON = 0.08                         # AUGMENTÉ de 0.03 → 0.08 (proche test 0.1)
CLEAN_RATIO = 0.7                      # RÉDUIT de 0.8 → 0.7 (moins de clean)
FGSM_RATIO = 0.3                       # AUGMENTÉ de 0.2 → 0.3 (plus de FGSM)
PGD_RATIO = 0.0                        # PAS DE PGD (focus FGSM)
PGD_STEPS = 0                          # Pas utilisé

# Early stopping
EARLY_STOPPING_PATIENCE = 8            # Arrêt si pas d'amélioration

# Créer le dossier de sortie
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

# Transformations - Augmentation modérée
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.4),
    transforms.RandomRotation(8),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

print("\n🛡️ CONFIGURATION FGSM OPTIMIZED:")
print(f"  Strategy: FGSM UNIQUEMENT avec epsilon proche du test")
print(f"  Epsilon: {EPSILON} (vs 0.1 au test = ratio {EPSILON/0.1:.1%})")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Balance: {CLEAN_RATIO*100:.0f}% clean + {FGSM_RATIO*100:.0f}% FGSM")
print(f"  Epochs: {NUM_EPOCHS} (avec early stopping patience={EARLY_STOPPING_PATIENCE})")
print(f"\n  ✅ Optimisé pour ROBUSTESSE FGSM maximale")
print(f"  🎯 Objectif: 75-82% robustesse FGSM, 89-91% val acc")



In [ ]:
# Créer les datasets
print("📦 Chargement des datasets...\n")

train_dataset = DangerousObjectDataset(DATA_DIR, 'train', train_transform)
val_dataset = DangerousObjectDataset(DATA_DIR, 'val', val_transform)
test_dataset = DangerousObjectDataset(DATA_DIR, 'test', val_transform)

# Créer les dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         num_workers=2, drop_last=True)  # drop_last pour stabilité
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n✅ Dataloaders créés!")

### 🔒 Zone 1: Application des mesures de sécurité des données

Avant de charger les datasets, nous appliquons:
1. **DataVerifier**: Tests statistiques sur la qualité
2. **PoisoningDetector**: Détection par clustering DBSCAN
3. **Quarantine automatique**: Isolation des échantillons suspects

In [ ]:
# Application des mesures Zone 1
train_data_path = os.path.join(DATA_DIR, 'train')
suspicious_files = apply_zone1_security(train_data_path, output_dir='data/quarantine')

print(f"\n📊 Résumé Zone 1:")
print(f"  Fichiers suspects détectés: {len(suspicious_files)}")
print(f"  Ces fichiers seront EXCLUS de l'entraînement")

In [ ]:
# Créer le modèle
print("\n🏗️ Création du modèle MobileNetV2 sécurisé...")
model = MobileNetV2Classifier(num_classes=2, pretrained=True)
model = model.to(device)

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Loss et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# Générateur d'exemples adversariaux
if ADVERSARIAL_TRAINING:
    adv_generator = AdversarialExampleGenerator(model, criterion, epsilon=EPSILON)
    print(f"\n🛡️ Générateur adversarial initialisé (epsilon={EPSILON})")

# Early Stopping pour éviter overfitting
class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"\n⏹️ Early stopping déclenché après {self.counter} epochs sans amélioration")
        else:
            self.best_loss = val_loss
            self.counter = 0

# Initialiser early stopping
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)
print(f"✅ Early stopping configuré (patience={EARLY_STOPPING_PATIENCE})")


# Créer les datasets avec filtrage Zone 1
print("📦 Chargement des datasets (avec exclusion fichiers suspects)...\n")

# IMPORTANT: Passer suspicious_files au dataset train uniquement
train_dataset = DangerousObjectDataset(DATA_DIR, 'train', train_transform, excluded_files=suspicious_files)
val_dataset = DangerousObjectDataset(DATA_DIR, 'val', val_transform)
test_dataset = DangerousObjectDataset(DATA_DIR, 'test', val_transform)

# Créer les dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         num_workers=2, drop_last=True)  # drop_last pour stabilité DP
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n✅ Dataloaders créés (avec filtrage Zone 1 appliqué)!")

In [ ]:
def train_epoch_adversarial(model, train_loader, criterion, optimizer, adv_generator, device):
    """Entraîne le modèle avec Adversarial Training"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc="Training (Adversarial)")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # 1. Loss sur données propres (50% du batch)
        clean_output = model(images)
        clean_loss = criterion(clean_output, labels)

        # 2. Adversarial Training - FGSM UNIQUEMENT (PAS DE PGD)
        if ADVERSARIAL_TRAINING:
            # Générer exemples FGSM
            fgsm_data = adv_generator.generate_fgsm(images, labels)
            fgsm_output = model(fgsm_data)
            fgsm_loss = criterion(fgsm_output, labels)


            # PAS DE PGD - trop agressif
            # total_loss avec FGSM uniquement: 80% clean + 20% FGSM
            total_loss = CLEAN_RATIO * clean_loss + FGSM_RATIO * fgsm_loss
        else:
            total_loss = clean_loss

        # Backpropagation
        total_loss.backward()

        # Gradient clipping pour stabilité
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Métriques (sur données propres)
        running_loss += total_loss.item() * images.size(0)
        _, predicted = clean_output.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': total_loss.item(), 'acc': 100. * correct / total})

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Évalue le modèle sur le validation set"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc



In [ ]:
# Entraînement
print("="*60)
print("🛡️ ENTRAÎNEMENT SÉCURISÉ AVEC ADVERSARIAL TRAINING")
print("="*60)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    # Train avec Adversarial Training
    train_loss, train_acc = train_epoch_adversarial(
        model, train_loader, criterion, optimizer, adv_generator, device
    )

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Update learning rate
    scheduler.step(val_loss)

    # Early stopping check
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f"\n⏹️ Arrêt anticipé à l'epoch {epoch}")
        break

    # Sauvegarder l'historique
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Afficher les résultats
    print(f"\n📊 Résultats Epoch {epoch}:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    # Sauvegarder le meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'config': {
                'adversarial_training': ADVERSARIAL_TRAINING,
                'epsilon': EPSILON,
                'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE
            },
            'timestamp': timestamp
        }, os.path.join(OUTPUT_DIR, 'best_secured_model.pth'))
        print(f"  ✓ Meilleur modèle sécurisé sauvegardé (val_acc: {val_acc:.2f}%)")

    # Sauvegarde périodique
    if (epoch) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, os.path.join(OUTPUT_DIR, f'secured_model_epoch_{epoch}_{timestamp}.pth'))

print("\n" + "="*60)
print("✅ ENTRAÎNEMENT SÉCURISÉ TERMINÉ")
print("="*60)


## 7. Évaluation finale sur le test set

In [ ]:
print("="*60)
print("🧪 ÉVALUATION FINALE SUR TEST SET")
print("="*60)

test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"\n📊 Résultats Test:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Acc:  {test_acc:.2f}%")

# Sauvegarder le modèle final
torch.save({
    'model_state_dict': model.state_dict(),
    'test_acc': test_acc,
    'test_loss': test_loss,
    'history': history,
    'config': {
        'adversarial_training': ADVERSARIAL_TRAINING,
        'epsilon': EPSILON,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE
    },
    'timestamp': timestamp
}, os.path.join(OUTPUT_DIR, 'final_secured_model.pth'))

# Sauvegarder l'historique
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n🎯 Meilleure validation accuracy: {best_val_acc:.2f}%")
print(f"🧪 Test accuracy finale: {test_acc:.2f}%")

## 8. Visualisation des résultats

### 🛡️ Zone 2: Application des mesures post-entraînement

Après l'entraînement, nous appliquons:
1. **Chiffrement AES-256-GCM**: Protection du modèle
2. **Signature RSA-4096**: Traçabilité et intégrité

In [ ]:
# Application des mesures Zone 2
best_model_path = os.path.join(OUTPUT_DIR, 'best_secured_model.pth')
apply_zone2_post_training(best_model_path, OUTPUT_DIR)

In [ ]:
# Générer les graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss (Adversarial)', linewidth=2)
ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax1.set_title('Secured Training Loss (Adversarial Training)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc (Clean)', linewidth=2)
ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
ax2.set_title('Secured Training Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 Graphiques sauvegardés: {OUTPUT_DIR}/training_history.png")

## 9. Téléchargement des résultats

Téléchargez les fichiers suivants pour les importer dans votre projet local:
- `best_secured_model.pth` - Meilleur modèle sécurisé
- `final_secured_model.pth` - Modèle final
- `training_history.json` - Historique d'entraînement
- `training_history.png` - Graphiques de performance

In [ ]:
import shutil
import os
from pathlib import Path
# IMPORTANT: Ré-importer files pour éviter écrasement par upload précédent
from google.colab import files as colab_files

# Créer le dossier temporaire pour l'archive
archive_temp_dir = 'secured_results_temp'
os.makedirs(archive_temp_dir, exist_ok=True)

print("📦 Préparation de l'archive des résultats sécurisés...")

# Copier les modèles et résultats
if os.path.exists(OUTPUT_DIR):
    for item in os.listdir(OUTPUT_DIR):
        src = os.path.join(OUTPUT_DIR, item)
        dst = os.path.join(archive_temp_dir, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
    print("  ✓ Modèles et historiques copiés")

# ZONE 1: Ajouter le dossier quarantine s'il existe
quarantine_base_dir = 'data/quarantine'
total_quarantined = 0

if os.path.exists(quarantine_base_dir):
    quarantine_dest = os.path.join(archive_temp_dir, 'quarantine')
    shutil.copytree(quarantine_base_dir, quarantine_dest)
    
    # Compter le nombre de fichiers en quarantaine
    for root, dirs, file_list in os.walk(quarantine_dest):
        total_quarantined += len([f for f in file_list if f.endswith('.jpg')])
    
    print(f"  ✓ Quarantine copiée ({total_quarantined} images suspectes)")
else:
    print("  ℹ️  Pas de données en quarantaine")

# Créer le ZIP
shutil.make_archive('secured_results', 'zip', archive_temp_dir)

# Nettoyer le dossier temporaire
shutil.rmtree(archive_temp_dir)

print("\n📊 Contenu de l'archive:")
print("  - best_secured_model.pth")
print("  - final_secured_model.pth")
print("  - training_history.json")
print("  - training_history.png")
if os.path.exists(quarantine_base_dir):
    print(f"  - quarantine/ ({total_quarantined} images suspectes + rapports)")

# Télécharger l'archive avec le module correctement importé
print("\n⬇️ Téléchargement de secured_results.zip...")
colab_files.download('secured_results.zip')

print("\n✅ Téléchargement terminé!")
print("\n📂 Pour importer dans votre projet:")
print("   1. Extraire secured_results.zip")
print("   2. Copier le contenu dans models/secured/")
if os.path.exists(quarantine_base_dir):
    print("   3. Copier quarantine/ dans data/quarantine/ pour inspection")

### 📋 Récapitulatif des fichiers de sécurité créés

Votre modèle sécurisé est maintenant protégé par toutes les mesures Zone 1 & Zone 2:

In [ ]:
# Liste tous les fichiers de sécurité créés
print("="*60)
print("📁 FICHIERS DE SÉCURITÉ CRÉÉS")
print("="*60)

security_files = {
    "Zone 1 - Sécurité des Données": [
        "data/quarantine/train_TIMESTAMP/ (images suspectes isolées)",
        "data/quarantine/train_TIMESTAMP/report.json (rapport détection)"
    ],
    "Zone 2 - Entraînement Sécurisé": [
        "models/secured/best_secured_model.pth (modèle principal)",
        "models/secured/best_secured_model_encrypted.enc (AES-256-GCM)",
        "models/secured/best_secured_model_encrypted_metadata.json",
        "models/secured/best_secured_model_signature.bin (RSA-4096)",
        "models/secured/best_secured_model_public_key.pem",
        "models/secured/best_secured_model_private_key.pem (🔐 SÉCURISÉE)"
    ],
    "Résultats d'entraînement": [
        "models/secured/training_history.json",
        "models/secured/training_history.png"
    ]
}

for category, files in security_files.items():
    print(f"\n🔒 {category}:")
    for file in files:
        print(f"  ✓ {file}")

print("\n" + "="*60)
print("✅ Architecture de sécurisation complète appliquée!")
print("   - Zone 1: 100% (DataVerifier, PoisoningDetector, Quarantine)")
print("   - Zone 2: 100% (Adversarial Training, Encryption AES-256, Signatures RSA-4096)")
print("="*60)

## 10. Résumé des résultats

In [ ]:
print("="*60)
print("📋 RÉSUMÉ DE L'ENTRAÎNEMENT SÉCURISÉ")
print("="*60)
print(f"\n🏗️ Architecture: MobileNetV2 (identique au baseline)")
print(f"🛡️ Sécurité:")
print(f"   - Adversarial Training: {ADVERSARIAL_TRAINING}")
print(f"   - Epsilon: {EPSILON}")
print(f"   - Attaques: FGSM (25%) + PGD (25%)")
print(f"   - Gradient clipping: Oui (max_norm=1.0)")
print(f"\n📊 Dataset:")
print(f"   - Train: {len(train_dataset)} images")
print(f"   - Val:   {len(val_dataset)} images")
print(f"   - Test:  {len(test_dataset)} images")
print(f"\n⚙️ Hyperparamètres:")
print(f"   - Epochs: {epoch}")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"   - Weight decay: {WEIGHT_DECAY}")
print(f"\n🎯 Performance:")
print(f"   - Meilleure val accuracy: {best_val_acc:.2f}%")
print(f"   - Test accuracy finale:  {test_acc:.2f}%")
print(f"   - Test loss finale:      {test_loss:.4f}")
print(f"\n🖥️ Device utilisé: {device}")
print("="*60)

print("\n⚠️ NOTE IMPORTANTE:")
print("L'accuracy peut être légèrement inférieure au baseline (~2-5%)")
print("C'est normal: trade-off entre accuracy standard et robustesse adversariale")
print("\nLe gain se verra lors des tests d'attaques adversariales!")

## Prochaines étapes

1. ✅ Télécharger `secured_results.zip`
2. Extraire le contenu dans votre projet local: `models/secured/`
3. **Tester la robustesse** avec `attack_secured.py`
4. **Comparer avec le baseline**: Robustesse vs Accuracy

---

### Différences attendues

| Métrique | Baseline | Secured (Adversarial Training) |
|----------|----------|--------------------------------|
| Clean Accuracy | 95-98% | 92-96% (-2 à -5%) |
| Adversarial Accuracy (FGSM) | 50% | 75-85% (+25 à +35%) |
| Adversarial Accuracy (PGD) | 30% | 60-75% (+30 à +45%) |
| Robustness Degradation | 47% | 10-20% (-27 à -37%) |

**Trade-off:**
- Légère baisse d'accuracy sur données propres
- **Grosse amélioration** de robustesse contre attaques adversariales